# Bigfoot reports: geography and seasonality
This analysis uses the course BFRO reports dataset, read directly from its public URL. Counts describe reports in this file, not verified sightings or population-adjusted rates.

In [1]:
import pandas as pd
import altair as alt
from pathlib import Path
DATA_URL = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
raw = pd.read_csv(DATA_URL)
print(f"Loaded {len(raw):,} rows")

Loaded 4,747 rows


## Clean and summarize
Trim categorical fields, retain records with state names, and choose the 15 states with the most records across all classifications. Do not drop duplicate-looking reports without evidence they are duplicates. Season is taken from the supplied season field, rather than inferred from dates, which are missing for some reports.

In [2]:
df = raw.copy()
for col in ["state", "season", "classification"]:
    df[col] = df[col].astype("string").str.strip()
df = df[df["state"].notna() & df["state"].ne("")].copy()
df["classification"] = df["classification"].fillna("Unspecified").replace("", "Unspecified")
top_states = df["state"].value_counts().head(15).index.tolist()
subset = df[df["state"].isin(top_states)].copy()
classes = sorted(subset["classification"].unique().tolist())
# Include zeros so a state remains visible when a class has no reports.
idx = pd.MultiIndex.from_product([top_states, classes], names=["state", "classification"])
by_class = subset.groupby(["state", "classification"]).size().reindex(idx, fill_value=0).rename("reports").reset_index()
all_counts = subset.groupby("state").size().rename("reports").reset_index().assign(classification="All")
state_counts = pd.concat([all_counts, by_class], ignore_index=True)
print(f"Retained {len(df):,} rows with a state; top 15 contain {len(subset):,} reports.")

Retained 4,747 rows with a state; top 15 contain 3,237 reports.


## Plot 1: state report counts
A classification dropdown filters precomputed counts. States are selected using all reports so the comparison set remains stable; their order changes with the selected counts. The teal color is constant and encodes no additional variable.

In [3]:
classification = alt.param(name="classification_filter", value="All",
    bind=alt.binding_select(options=["All"] + classes, name="Report classification: "))
chart1 = (alt.Chart(state_counts).add_params(classification)
    .transform_filter(alt.datum.classification == classification)
    .mark_bar(color="#167d8d", cornerRadiusEnd=3)
    .encode(x=alt.X("reports:Q", title="Number of reports", scale=alt.Scale(zero=True)),
            y=alt.Y("state:N", sort="-x", title=None),
            tooltip=[alt.Tooltip("state:N", title="State"), alt.Tooltip("classification:N", title="Classification"), alt.Tooltip("reports:Q", title="Reports", format=",")])
    .properties(width=650, height=440, title="Reports in the 15 most-reported states")
    .configure_view(stroke=None))
chart1

Altair chart (interactive when embedded on the webpage).

## Plot 2: seasonal composition
Exclude unrecognized or missing seasons only from this plot. Complete every state-season combination with zero counts, then calculate percentages within each state using only reports with a recognized season. The heatmap compares composition rather than raw totals, and includes both values in tooltips.

In [4]:
seasons = ["Winter", "Spring", "Summer", "Fall"]
season_valid = subset[subset["season"].isin(seasons)]
sidx = pd.MultiIndex.from_product([top_states, seasons], names=["state", "season"])
season_counts = season_valid.groupby(["state", "season"]).size().reindex(sidx, fill_value=0).rename("reports").reset_index()
season_counts["state_total"] = season_counts.groupby("state")["reports"].transform("sum")
season_counts["share"] = season_counts["reports"].div(season_counts["state_total"].replace(0, float("nan")))
print(f"Seasonal plot excludes {len(subset) - len(season_valid):,} reports with missing or unrecognized seasons.")
chart2 = (alt.Chart(season_counts).mark_rect(stroke="white", strokeWidth=2)
    .encode(x=alt.X("season:O", sort=seasons, title="Reported season"),
            y=alt.Y("state:N", sort=top_states, title=None),
            color=alt.Color("share:Q", title="Share within state", scale=alt.Scale(scheme="blues", domain=[0, 1]), legend=alt.Legend(format=".0%")),
            tooltip=[alt.Tooltip("state:N", title="State"), alt.Tooltip("season:O", title="Season"), alt.Tooltip("reports:Q", title="Reports", format=","), alt.Tooltip("state_total:Q", title="Known-season reports", format=","), alt.Tooltip("share:Q", title="Share", format=".1%")])
    .properties(width=550, height=440, title="Seasonal share of reports within each state")
    .configure_view(stroke=None))
chart2

Seasonal plot excludes 63 reports with missing or unrecognized seasons.


Altair chart (interactive when embedded on the webpage).

## Export
Run from the repository root, or from its notebooks folder. JSON specifications contain the small aggregated tables inline, so the webpage does not need to fetch the original CSV.

In [5]:
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assets = root / "assets"
assets.mkdir(exist_ok=True)
chart1.save(str(assets / "state_reports.json"))
chart2.save(str(assets / "season_heatmap.json"))
chart1.save(str(assets / "state_reports.html"))
chart2.save(str(assets / "season_heatmap.html"))